# Tahap 4 — Case Solution Reuse (IMPROVED)

Mengeksekusi Tahap 4 dari pipeline CBR. Fokus utama adalah "Reusing" (menggunakan kembali) solusi dari kasus-kasus terdahulu yang paling relevan (hasil dari *Retrieval*) untuk memprediksi hasil/putusan dari kasus baru.

### Perbaikan:
- **Sinkronisasi preprocessing** dengan perbaikan di Tahap 1
- **Similarity threshold** pada saat retrieval
- **Retrieval explanation** per query (menjelaskan alasan perceraian dari kasus yang ditarik)

## 1. Import Library

In [1]:
import os
import re
import json
import joblib
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm

import nltk
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

print("Semua library untuk Tahap 4 siap!")

Semua library untuk Tahap 4 siap!


## 2. Load Dataset dan Model dari Tahap Sebelumnya

In [2]:
base_path = "../"
csv_path = os.path.join(base_path, "data/processed/cases.csv")
models_dir = os.path.join(base_path, "models")
results_dir = os.path.join(base_path, "data/results")
os.makedirs(results_dir, exist_ok=True)

# Load Dataset
df_cases = pd.read_csv(csv_path)
df_cases = df_cases.dropna(subset=['text_full', 'amar_putusan'])

# Load Models
tfidf_vectorizer = joblib.load(os.path.join(models_dir, "tfidf_vectorizer.pkl"))
svm_model = joblib.load(os.path.join(models_dir, "svm_classifier.pkl"))

# Persiapkan matriks dokumen
X_all_tfidf = tfidf_vectorizer.transform(df_cases['text_full'])

print(f"Dataset termuat: {len(df_cases)} kasus.")
print("Model TF-IDF dan SVM berhasil diload!")

Dataset termuat: 62 kasus.
Model TF-IDF dan SVM berhasil diload!


## 3. Implementasi `retrieve()` (IMPROVED)

Menggunakan *Similarity Threshold* agar tidak meretrieve dokumen noise.

In [3]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stem_cache = {}

stop_words_id = set(stopwords.words('indonesian'))
custom_stopwords = {
    'pengadilan', 'hakim', 'perkara', 'putusan', 'republik', 'indonesia',
    'agama', 'mahkamah', 'agung', 'direktori', 'majelis', 'sidang',
    'panitera', 'bahwa', 'yang', 'dan', 'di', 'dari', 'pada', 'untuk',
    'dengan', 'ini', 'itu', 'adalah', 'oleh', 'kepada', 'sebagai',
    'tersebut', 'telah', 'akan', 'dapat', 'dalam', 'atau'
}
stop_words_id.update(custom_stopwords)

def clean_query(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [w for w in tokens if len(w) > 1 and w not in stop_words_id]
    stemmed = []
    for t in tokens:
        if t not in stem_cache:
            stem_cache[t] = stemmer.stem(t)
        s = stem_cache[t]
        if len(s) > 1:
            stemmed.append(s)
    return ' '.join(stemmed)

SIMILARITY_THRESHOLD = 0.01

def retrieve(query: str, k: int = 5):
    query_clean = clean_query(query)
    query_vec = tfidf_vectorizer.transform([query_clean])
    
    similarity_scores = cosine_similarity(query_vec, X_all_tfidf).flatten()
    top_k_indices = similarity_scores.argsort()[-k:][::-1]
    
    results = []
    for idx in top_k_indices:
        score = similarity_scores[idx]
        if score >= SIMILARITY_THRESHOLD:
            case = df_cases.iloc[idx]
            results.append({
                "case_id": case["case_id"],
                "similarity_score": round(score, 4),
                "label_solusi": case["label"],
                "alasan": case.get("alasan_perceraian", "-")
            })
    return results

## 4. Algoritma Prediksi (Majority & Weighted)

Terdapat dua algoritma *voting*:
- **Majority Voting**: Mencari label terbanyak (modus) di top-k.
- **Weighted Similarity**: Menjumlahkan nilai kemiripan pada masing-masing label.

In [4]:
def predict_outcome(query: str, method: str = "weighted", k: int = 5):
    top_k = retrieve(query, k=k)
    
    # Jika tidak ada kasus mirip yang melebihi threshold
    if not top_k:
        return {
            "query": query, 
            "predicted_solution": "Tidak ada solusi (No Matches)", 
            "details": {},
            "top_5_case_ids": [],
            "explanations": []
        }
        
    solutions = [res['label_solusi'] for res in top_k]
    scores = [res['similarity_score'] for res in top_k]
    case_ids = [res['case_id'] for res in top_k]
    explanations = [f"{res['case_id']} ({res['similarity_score']}): {res['alasan']}" for res in top_k]
    
    # -- MAJORITY VOTING --
    majority_pred = Counter(solutions).most_common(1)[0][0]
    
    # -- WEIGHTED SIMILARITY VOTING --
    weights = {}
    for r in top_k:
        lbl = r['label_solusi']
        weights[lbl] = weights.get(lbl, 0.0) + r['similarity_score']
    weighted_pred = max(weights, key=weights.get)
    
    final_pred = weighted_pred if method == "weighted" else majority_pred
    
    # Prediksi SVM
    query_vec = tfidf_vectorizer.transform([clean_query(query)])
    svm_pred = svm_model.predict(query_vec)[0]
    
    return {
        "query": query,
        "predicted_solution": final_pred,
        "top_5_case_ids": case_ids,
        "explanations": explanations,
        "details": {
            "similarity_scores": scores,
            "top_k_labels": solutions,
            "majority_voting_result": majority_pred,
            "weighted_voting_scores": weights,
            "svm_prediction": svm_pred
        }
    }

## 5. Simulasi Prediksi Kasus Baru

In [5]:
queries_uji = [
    "suami tidak memberi nafkah selama 2 tahun",
    "terjadi perselisihan terus menerus",
    "penggugat meninggalkan rumah",
    "terjadi kekerasan rumah tangga dipukul",
    "suami cacat dan istri tidak tahan karena ekonomi"
]

print("=== SIMULASI PREDIKSI KASUS BARU (REUSE) ===\n")
hasil_prediksi = []

for i, q in enumerate(queries_uji):
    q_id = f"Q{i+1}"
    res = predict_outcome(q, method="weighted", k=5)
    
    print(f"Query [{q_id}]: '{q}'")
    print(f"-> Top Kasus Terkait: {res['top_5_case_ids']}")
    if res['top_5_case_ids']:
        print(f"-> Penjelasan: {res['explanations'][0]}")
    print(f"-> Prediksi Akhir: **{res['predicted_solution'].upper()}**\n")
    
    hasil_prediksi.append({
        "query_id": q_id,
        "query": q,
        "predicted_solution": res['predicted_solution'],
        "top_case_ids": ", ".join(res['top_5_case_ids']),
        "similarity_scores": ", ".join([str(s) for s in res['details'].get('similarity_scores', [])]),
        "svm_prediction": res['details'].get('svm_prediction', '-')
    })

=== SIMULASI PREDIKSI KASUS BARU (REUSE) ===

Query [Q1]: 'suami tidak memberi nafkah selama 2 tahun'
-> Top Kasus Terkait: ['case_059', 'case_010', 'case_024', 'case_023', 'case_055']
-> Penjelasan: case_059 (0.0571): pertengkaran, ekonomi, meninggalkan, kekerasan, selingkuh
-> Prediksi Akhir: **DIKABULKAN**



Query [Q2]: 'terjadi perselisihan terus menerus'
-> Top Kasus Terkait: ['case_060', 'case_032', 'case_006', 'case_022', 'case_016']
-> Penjelasan: case_060 (0.0439): pertengkaran, meninggalkan
-> Prediksi Akhir: **DIKABULKAN**



Query [Q3]: 'penggugat meninggalkan rumah'
-> Top Kasus Terkait: ['case_022', 'case_050', 'case_016', 'case_036', 'case_010']
-> Penjelasan: case_022 (0.0604): pertengkaran, meninggalkan, kekerasan, selingkuh
-> Prediksi Akhir: **DIKABULKAN**

Query [Q4]: 'terjadi kekerasan rumah tangga dipukul'
-> Top Kasus Terkait: ['case_002', 'case_012', 'case_007', 'case_035', 'case_033']
-> Penjelasan: case_002 (0.077): pertengkaran, ekonomi, kekerasan, selingkuh
-> Prediksi Akhir: **DIKABULKAN**

Query [Q5]: 'suami cacat dan istri tidak tahan karena ekonomi'
-> Top Kasus Terkait: ['case_052', 'case_023', 'case_008', 'case_014', 'case_040']
-> Penjelasan: case_052 (0.0983): pertengkaran, ekonomi
-> Prediksi Akhir: **DIKABULKAN**



## 6. Export Prediksi

In [6]:
df_predictions = pd.DataFrame(hasil_prediksi)
csv_out_path = os.path.join(results_dir, "predictions.csv")
df_predictions.to_csv(csv_out_path, index=False)

print(f"Berhasil menyimpan prediksi ke: {csv_out_path}")

# Export JSON format untuk evaluasi spesifik jika dibutuhkan oleh Tahap 5
json_out_path = os.path.join(results_dir, "predictions_detail.json")
with open(json_out_path, "w", encoding="utf-8") as f:
    json.dump(hasil_prediksi, f, indent=4)

print("\nTAHAP 4 (CASE SOLUTION REUSE) SELESAI DAN SIAP DILANJUTKAN KE TAHAP 5!")

Berhasil menyimpan prediksi ke: ../data/results\predictions.csv

TAHAP 4 (CASE SOLUTION REUSE) SELESAI DAN SIAP DILANJUTKAN KE TAHAP 5!
